# Analise Exploratoria - Camada SILVER
## E-commerce Brasileiro (Dataset Moderno)

---

### Arquitetura de Dados - Camada SILVER

**Contexto:**  
Esta analise opera sobre a **Camada SILVER** da arquitetura medalhao, onde os dados modernos foram:
- Integrados a partir do dataset transacional de vendas e enriquecidos por dimensao de produtos
- Padronizados para consumo analitico (`fact_sales_modern`)
- Validados quanto a qualidade, consistencia e duplicidade de chave
- Preparados para integracao posterior com a camada historica

**Fonte de Dados:**  
`fact_sales_modern.parquet` (preferencial) ou `fact_sales_modern.csv`

---

### Objetivos da Analise

1. Validar qualidade e integridade da tabela consolidada moderna
2. Confirmar consistencia de chave de negocio (`order_id + order_item_id`)
3. Explorar comportamento de vendas, status e categorias
4. Verificar distribuicoes e outliers de ticket
5. Consolidar prontidao da Silver para etapa de integracao

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

plt.style.use("seaborn-v0_8-darkgrid")

SILVER_DIR = Path(".")
parquet_path = SILVER_DIR / "fact_sales_modern.parquet"
csv_path = SILVER_DIR / "fact_sales_modern.csv"

fact_sales_modern = None
source_used = None

if parquet_path.exists():
    try:
        fact_sales_modern = pd.read_parquet(parquet_path)
        source_used = parquet_path.name
    except Exception as parquet_err:
        print(f"Falha ao ler Parquet ({parquet_err}). Tentando CSV...")

if fact_sales_modern is None and csv_path.exists():
    fact_sales_modern = pd.read_csv(csv_path, encoding="utf-8")
    source_used = csv_path.name

if fact_sales_modern is None:
    raise FileNotFoundError("Nenhum arquivo fact_sales_modern valido encontrado em Data Layer/silver")

fact_sales_modern["order_date"] = pd.to_datetime(fact_sales_modern["order_date"], errors="coerce")
fact_sales_modern["gross_revenue"] = pd.to_numeric(fact_sales_modern["gross_revenue"], errors="coerce")
fact_sales_modern["unit_price"] = pd.to_numeric(fact_sales_modern["unit_price"], errors="coerce")

print(f"Fonte carregada: {source_used}")
print(f"Dimensoes: {fact_sales_modern.shape[0]:,} linhas x {fact_sales_modern.shape[1]} colunas")

print("\nHead:")
display(fact_sales_modern.head())

print("\nInfo:")
fact_sales_modern.info()

Fonte: fact_sales_modern.parquet
Dimensoes: 128,975 linhas x 32 colunas


,order_id,order_item_id,order_date,order_year,order_month,order_year_month,order_status,courier_status,fulfillment,sales_channel,ship_service_level,sku,style,asin,category,size,color,quantity,unit_price,gross_revenue,currency,ship_city,ship_state,ship_postal_code,ship_country,promotion_ids,is_b2b,fulfilled_by,stock_available,is_cancelled,is_returned,is_delivered
0,171-0000547-8192359,1,2022-06-07,2022,6,2022-06,Shipped,Shipped,Amazon,Amazon.in,Expedited,JNE2032-KR-205-XL,JNE2032,B0768J7VQ1,kurta,XL,Teal,1,301.0,301.0,INR,PIMPRI CHINCHWAD,MAHARASHTRA,412101,IN,<NA>,False,<NA>,6.0,False,False,False
1,171-0000902-4490745,1,2022-06-09,2022,6,2022-06,Cancelled,Unshipped,Amazon,Amazon.in,Expedited,J0097-KR-M,J0097,B08BJT3PSM,kurta,M,Indigo,1,544.0,544.0,INR,Howrah,WEST BENGAL,711104,IN,<NA>,False,<NA>,4.0,True,False,False
2,171-0001409-6228339,1,2022-06-07,2022,6,2022-06,Shipped,Shipped,Amazon,Amazon.in,Expedited,JNE3440-KR-N-XS,JNE3440,B09HMY3YLT,kurta,XS,<NA>,1,422.0,422.0,INR,KODAD,TELANGANA,508206,IN,<NA>,False,<NA>,NaN,False,False,False
3,171-0003082-5110755,1,2022-05-04,2022,5,2022-05,Shipped,Shipped,Amazon,Amazon.in,Expedited,SET291-KR-PP-L,SET291,B099NJKJ54,Set,L,Teal,1,563.0,563.0,INR,GREATER NOIDA,UTTAR PRADESH,201306,IN,<NA>,False,<NA>,6.0,False,False,False
4,171-0003738-2052324,1,2022-04-03,2022,4,2022-04,Shipped,Shipped,Amazon,Amazon.in,Expedited,JNE3440-KR-N-XS,JNE3440,B09HMY3YLT,kurta,XS,<NA>,1,379.0,379.0,INR,FIROZABAD,UTTAR PRADESH,283203,IN,<NA>,False,<NA>,NaN,False,False,False


### Dicionario de Dados — fact_sales_modern

Mapeamento de colunas, tipos e origem de cada atributo da tabela moderna consolidada.

In [ ]:
origem_map = {
    "order_id": "amazon_sale_report",
    "order_item_id": "silver (derivado por ordenacao no pedido)",
    "order_date": "amazon_sale_report",
    "order_year": "silver (derivado de order_date)",
    "order_month": "silver (derivado de order_date)",
    "order_year_month": "silver (derivado de order_date)",
    "order_status": "amazon_sale_report",
    "courier_status": "amazon_sale_report",
    "fulfillment": "amazon_sale_report",
    "sales_channel": "amazon_sale_report",
    "ship_service_level": "amazon_sale_report",
    "sku": "amazon_sale_report",
    "style": "amazon_sale_report/sale_report",
    "asin": "amazon_sale_report",
    "category": "amazon_sale_report/sale_report",
    "size": "amazon_sale_report/sale_report",
    "color": "sale_report",
    "quantity": "amazon_sale_report",
    "unit_price": "silver (gross_revenue / quantity)",
    "gross_revenue": "amazon_sale_report (amount)",
    "currency": "amazon_sale_report",
    "ship_city": "amazon_sale_report",
    "ship_state": "amazon_sale_report",
    "ship_postal_code": "amazon_sale_report",
    "ship_country": "amazon_sale_report",
    "promotion_ids": "amazon_sale_report",
    "is_b2b": "amazon_sale_report",
    "fulfilled_by": "amazon_sale_report",
    "stock_available": "sale_report",
    "is_cancelled": "silver (derivado de order_status)",
    "is_returned": "silver (derivado de order_status)",
    "is_delivered": "silver (derivado de order_status)",
}

dict_df = pd.DataFrame({
    "Coluna": fact_sales_modern.columns,
    "Tipo": fact_sales_modern.dtypes.astype(str).values,
    "Nulos": fact_sales_modern.isnull().sum().values,
    "Nulos_%": (fact_sales_modern.isnull().mean() * 100).round(2).values,
    "Origem": [origem_map.get(c, "silver (calculado)") for c in fact_sales_modern.columns],
})

print(f"Dicionario da fact_sales_modern  —  {len(dict_df)} colunas")
display(dict_df)

## Analises Exploratorias de Validacao

Analises para validar consistencia da Silver moderna:
- Estatisticas descritivas
- Valores nulos
- Duplicidades de chave e linha
- Distribuicao de status
- Evolucao mensal de receita
- Top categorias por receita
- Distribuicao de ticket medio

### Contexto de Negocio

As analises abaixo verificam se os padroes da Silver moderna estao coerentes para integrar com a camada historica na Sprint 4.

In [ ]:
print("Estatisticas descritivas (numericas):")
display(fact_sales_modern.describe(include=[np.number]).T)

print("\nEstatisticas descritivas (categoricas):")
display(fact_sales_modern.describe(include=["object", "category", "bool"]).T)

nulls = fact_sales_modern.isnull().sum().sort_values(ascending=False)
nulls_pct = (nulls / len(fact_sales_modern) * 100).round(2)
nulls_df = pd.DataFrame({"null_count": nulls, "null_pct": nulls_pct})

print("\nTop colunas com nulos:")
display(nulls_df[nulls_df["null_count"] > 0].head(20))

key_dups = fact_sales_modern.duplicated(subset=["order_id", "order_item_id"]).sum()
row_dups = fact_sales_modern.duplicated().sum()

print(f"Duplicidades da chave (order_id + order_item_id): {key_dups:,}")
print(f"Duplicidades de linha completa: {row_dups:,}")

if key_dups == 0:
    print("Status da chave: OK")
else:
    print("Status da chave: ALERTA")

## Dashboard de Metricas — fact_sales_modern

Resumo executivo dos principais indicadores da tabela Silver moderna antes das analises graficas.

In [ ]:
print("=" * 62)
print("  RESUMO DA FACT_SALES_MODERN — CAMADA SILVER")
print("=" * 62)
print(f"  Pedidos..............: {fact_sales_modern['order_id'].nunique():,}")
print(f"  Itens vendidos.......: {len(fact_sales_modern):,}")
print(f"  SKUs.................: {fact_sales_modern['sku'].nunique():,}")
print(f"  Categorias...........: {fact_sales_modern['category'].nunique():,}")
print(f"  Estados de envio.....: {fact_sales_modern['ship_state'].nunique():,}")
print(f"  Receita total........: {fact_sales_modern['gross_revenue'].sum():,.2f}")
print(f"  Ticket medio/item....: {fact_sales_modern['unit_price'].mean():,.2f}")
print(f"  Taxa de cancelamento.: {fact_sales_modern['is_cancelled'].mean() * 100:.2f}%")
print(f"  Taxa de devolucao....: {fact_sales_modern['is_returned'].mean() * 100:.2f}%")
print("=" * 62)

In [ ]:
fa = fact_sales_modern.copy()

status_counts = fa['order_status'].fillna('UNKNOWN').value_counts().head(10)
receita_mensal = (
    fa.dropna(subset=['order_date'])
      .assign(ano_mes=lambda x: x['order_date'].dt.to_period('M').astype(str))
      .groupby('ano_mes', as_index=False)['gross_revenue'].sum()
      .sort_values('ano_mes')
)
top_cat = (
    fa.groupby('category', as_index=False)['gross_revenue']
      .sum()
      .sort_values('gross_revenue', ascending=False)
      .head(10)
)
ticket = fa['unit_price'].dropna()

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

axes[0, 0].bar(status_counts.index.astype(str), status_counts.values, color='#2E86AB')
axes[0, 0].set_title('Top Status de Pedido')
axes[0, 0].set_xlabel('Status')
axes[0, 0].set_ylabel('Quantidade')
axes[0, 0].tick_params(axis='x', rotation=35)

axes[0, 1].plot(receita_mensal['ano_mes'], receita_mensal['gross_revenue'], marker='o', linewidth=2, color='#F18F01')
axes[0, 1].set_title('Evolucao Mensal da Receita')
axes[0, 1].set_xlabel('Ano-Mes')
axes[0, 1].set_ylabel('Receita')
axes[0, 1].tick_params(axis='x', rotation=45)

axes[1, 0].barh(top_cat['category'].astype(str), top_cat['gross_revenue'], color='#C73E1D')
axes[1, 0].set_title('Top 10 Categorias por Receita')
axes[1, 0].set_xlabel('Receita')
axes[1, 0].set_ylabel('Categoria')

ticket_clip = ticket[ticket <= ticket.quantile(0.99)]
axes[1, 1].hist(ticket_clip, bins=50, color='#2A9D8F', alpha=0.85)
axes[1, 1].set_title('Distribuicao de Ticket por Item (ate P99)')
axes[1, 1].set_xlabel('Unit Price')
axes[1, 1].set_ylabel('Frequencia')

plt.tight_layout()
plt.show()

## Insights Encontrados

Com base nas analises graficas e nos indicadores da Silver moderna, destacam-se os seguintes pontos:

- **Status dos pedidos:** predominancia de registros enviados/entregues, com cancelamentos concentrados em uma parcela menor.
- **Concentracao de receita:** poucas categorias concentram a maior parte da receita, indicando curva de concentracao tipica de varejo.
- **Sazonalidade curta:** o periodo analisado (2022-03 a 2022-06) mostra dinamica mensal coerente para janela operacional curta.
- **Distribuicao de ticket:** assimetria a direita, com maioria dos itens em faixas menores e poucos outliers elevando a media.
- **Qualidade dos dados:** chave logica sem duplicidade e baixa incidencia de nulos em campos essenciais de faturamento e data.

## Fechamento da Silver

Esta secao consolida os principais indicadores de qualidade da `fact_sales_modern` para concluir a validacao tecnica da camada Silver moderna antes da integracao da Sprint 4.

In [ ]:
total_rows = len(fact_sales_modern)
total_cols = fact_sales_modern.shape[1]

key_dups = fact_sales_modern.duplicated(subset=['order_id', 'order_item_id']).sum()
key_is_valid = key_dups == 0

total_nulls = int(fact_sales_modern.isnull().sum().sum())
null_pct_global = (total_nulls / (total_rows * total_cols)) * 100

mem_mb = fact_sales_modern.memory_usage(deep=True).sum() / 1024 ** 2

print('Resumo da Qualidade dos Dados - Silver Modern')
print('-' * 62)
print(f'Quantidade de registros  : {total_rows:,}')
print(f'Quantidade de colunas    : {total_cols}')
print(f'Valores nulos totais     : {total_nulls:,} ({null_pct_global:.2f}% do dataset)')
print(f'Duplicidades da chave    : {key_dups:,}')
print(f"Integridade da chave     : {'OK' if key_is_valid else 'ALERTA'}")
print(f'Memoria utilizada        : {mem_mb:.2f} MB')

if key_is_valid:
    print('\nConclusao: a fact_sales_modern esta validada tecnicamente na camada Silver para seguir para validacao funcional da Sprint 3.')
else:
    print('\nConclusao: a fact_sales_modern requer ajuste de chave antes da integracao com a camada historica.')